In [50]:
import xarray as xr
import pandas as pd
from dask.diagnostics import ProgressBar
from pathlib import Path
import math

# -----------------------------
# CONFIG
# -----------------------------
%run Data_Config.ipynb
print(f"NetCDF input directory: {input_dir}")
print(f"Zarr output directory: {zarr_output_dir}")
print(f"Date range: {DATE_RANGE}")

station_batch_size = 10   # Number of stations processed at once
target_time = pd.date_range("1970-01-01", "2023-12-31", freq="h")  # example range
chunks = {"station": station_batch_size, "time": -1}  # tweak as needed

# -----------------------------
# PREPROCESS FUNCTION
# -----------------------------
def preprocess_one_file(path):
    """Open a single NetCDF file, reindex time, assign station coordinate."""
    ds = xr.open_dataset(path, chunks={"time": -1})
    if 'input_station_id' in ds:
        ds = ds.drop_vars('input_station_id')
    
    # Reindex time to common target
    ds = ds.reindex(time=target_time)
    
    # Assign station ID (adjust if your attribute key differs)
    station_id = ds.attrs.get("station_id", Path(path).stem)
    ds = ds.assign_coords(station=("station", [station_id]))
    
    return ds



# -----------------------------
# FIX CHUNK SIZES
# -----------------------------
def ensure_uniform_chunks(ds, station_chunk=1, time_chunk=-1):
    """
    Force uniform chunks for station and time, store all other dims as a single chunk.
    """
    chunk_map = {}
    for dim, size in ds.dims.items():
        if dim == "station":
            chunk_map[dim] = station_chunk
        elif dim == "time":
            chunk_map[dim] = time_chunk
        else:
            # Small or flag-like dims → store whole
            chunk_map[dim] = size
    return ds.chunk(chunk_map)


# ----------------------------
# CLEAN FILLVALUE ATTRIBUTES
# ----------------------------
def clean_fillvalue_attrs(ds, zarr_store_path=None):
    import numpy as np
    zarr_fills = {}
    if zarr_store_path is not None and Path(zarr_store_path).exists():
        ds_zarr = xr.open_zarr(zarr_store_path)
        for var in ds_zarr.data_vars:
            zarr_fills[var] = ds_zarr[var].encoding.get('_FillValue')
    for var in ds.data_vars:
        # Remove missing_value from encoding
        ds[var].encoding.pop('missing_value', None)
        # Align _FillValue with Zarr store if possible
        if var in zarr_fills and zarr_fills[var] is not None:
            # Coerce type to match variable dtype
            dtype = ds[var].dtype
            fill = zarr_fills[var]
            if fill is not None and not (isinstance(fill, float) and np.isnan(fill)):
                ds[var].encoding['_FillValue'] = dtype.type(fill)
            else:
                ds[var].encoding['_FillValue'] = np.nan
        # Otherwise, ensure _FillValue is set (choose your default)
        elif '_FillValue' not in ds[var].encoding:
            # Use a sensible default for the dtype
            dtype = ds[var].dtype
            if np.issubdtype(dtype, np.floating):
                ds[var].encoding['_FillValue'] = dtype.type(-999.0)
            else:
                ds[var].encoding['_FillValue'] = dtype.type(-999)
    return ds

# -----------------------------
# DROP FILLVALUE ATTRIBUTES
# -----------------------------
def drop_fillvalue_attrs(ds):
    for var in ds.data_vars:
        ds[var].encoding.pop('missing_value', None)
        ds[var].encoding.pop('_FillValue', None)
    return ds

# -----------------------------
# MAIN BATCHING LOOP
# -----------------------------
files = sorted(input_dir.glob("*.nc"))
num_batches = 2 #math.ceil(len(files) / station_batch_size)
start = 0
stop = 1

for i in range(num_batches):
    batch_files = files[i*station_batch_size : (i+1)*station_batch_size]
    print(f"Processing batch {i+1}/{num_batches} ({len(batch_files)} files)")
    
    batch_datasets = []
    for f in batch_files:
        batch_datasets.append(preprocess_one_file(f))
    
    # Concatenate stations in this batch
    batch_ds = xr.concat(batch_datasets, dim="station")
    
    # Ensure uniform chunks before writing
    batch_ds = ensure_uniform_chunks(batch_ds)

    batch_ds = drop_fillvalue_attrs(batch_ds)

    with ProgressBar():
        if i == 0:
            try:
                batch_ds.to_zarr(zarr_output_dir, mode="w")
                print("Initial Zarr store created.")
            except Exception as e:
                print(f"Error creating Zarr store: {e}")
        else:
            try:
                batch_ds.to_zarr(zarr_output_dir, mode="a", append_dim="station")
            except Exception as e:
                print(f"Error appending to Zarr: {e}")
    # Free memory
    del batch_ds, batch_datasets



NetCDF input directory: /Users/joelmiller/HadISD_data/netcdf
Zarr output directory: /Users/joelmiller/HadISD_data/zarr
Date range: ('1970-01-01T00', '2023-12-31T23')
Processing batch 1/2 (10 files)


/var/folders/v1/7wypnx5x11qgv022zwx5p3g00000gn/T/ipykernel_2229/1391177064.py:47: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for dim, size in ds.dims.items():
<frozen _collections_abc>:899: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
/var/folders/v1/7wypnx5x11qgv022zwx5p3g00000gn/T/ipykernel_2229/1391177064.py:126: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  batch_ds.to_zarr(zarr_output_dir, mode="w")
/var/folders/v1/7wypnx5x11qgv022zwx5p3g00000gn/T/ipykernel_2229/1391177064.py:126: SerializationWarning

[########################################] | 100% Completed | 5.68 ss
Initial Zarr store created.
Processing batch 2/2 (10 files)


/var/folders/v1/7wypnx5x11qgv022zwx5p3g00000gn/T/ipykernel_2229/1391177064.py:47: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for dim, size in ds.dims.items():
<frozen _collections_abc>:899: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
/var/folders/v1/7wypnx5x11qgv022zwx5p3g00000gn/T/ipykernel_2229/1391177064.py:132: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  batch_ds.to_zarr(zarr_output_dir, mode="a", append_dim="station")
/var/folders/v1/7wypnx5x11qgv022zwx5p3g00000gn/T/ipykernel_2229/1391177064.py:132

[########################################] | 100% Completed | 5.25 ss


In [32]:
ds0 = batch_ds_0
ds0

<xarray.Dataset> Size: 4GB
Dimensions:                (station: 10, time: 473329, test: 71, flagged: 19,
                            reporting_v: 19, reporting_t: 1116, reporting_2: 2,
                            coordinate_length: 1)
Coordinates:
  * time                   (time) datetime64[ns] 4MB 1970-01-01 ... 2023-12-31
    longitude              (station, coordinate_length) float64 80B dask.array<chunksize=(1, 1), meta=np.ndarray>
    latitude               (station, coordinate_length) float64 80B dask.array<chunksize=(1, 1), meta=np.ndarray>
    elevation              (station, coordinate_length) float64 80B dask.array<chunksize=(1, 1), meta=np.ndarray>
  * station                (station) <U12 480B '010010-99999' ... '010260-99999'
Dimensions without coordinates: test, flagged, reporting_v, reporting_t,
                                reporting_2, coordinate_length
Data variables: (12/26)
    station_id             (station) |S12 120B dask.array<chunksize=(1,), meta=np.ndarray>
    temperatures           (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    dewpoints              (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    slp                    (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    stnlp                  (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    windspeeds             (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    ...                     ...
    cloud_base             (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    wind_gust              (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    past_sigwx1            (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    quality_control_flags  (station, time, test) float64 3GB dask.array<chunksize=(1, 473329, 71), meta=np.ndarray>
    flagged_obs            (station, time, flagged) float64 719MB dask.array<chunksize=(1, 473329, 19), meta=np.ndarray>
    reporting_stats        (station, reporting_v, reporting_t, reporting_2) float64 3MB dask.array<chunksize=(1, 19, 1116, 2), meta=np.ndarray>
Attributes: (12/39)
    title:                       HadISD
    institution:                 Met Office Hadley Centre, Exeter, UK
    source:                      HadISD data product
    references:                  Dunn, 2019, Met Office Hadley Centre Technic...
    creator_name:                Robert Dunn
    creator_url:                 www.metoffice.gov.uk
    ...                          ...
    station_information:         Where station is a composite the station id ...
    Conventions:                 CF-1.6
    Metadata_Conventions:        Unidata Dataset Discovery v1.0, CF Discrete ...
    featureType:                 timeSeries
    processing_date:             08-Jan-2024
    history:                     Created by mk_netcdf_files.py \nDuplicate Mo...

In [33]:
for var in ds0.data_vars:
    mv = ds0[var].attrs.get("missing_value")
    fv = ds0[var].encoding.get("_FillValue")
    if mv is not None or fv is not None:
        print(f"{var}: missing_value={mv}, _FillValue={fv}")

temperatures: missing_value=None, _FillValue=-1e+30
dewpoints: missing_value=None, _FillValue=-1e+30
slp: missing_value=None, _FillValue=-1e+30
stnlp: missing_value=None, _FillValue=-1e+30
windspeeds: missing_value=None, _FillValue=-1e+30
winddirs: missing_value=None, _FillValue=-999
total_cloud_cover: missing_value=None, _FillValue=-999
low_cloud_cover: missing_value=None, _FillValue=-999
mid_cloud_cover: missing_value=None, _FillValue=-999
high_cloud_cover: missing_value=None, _FillValue=-999
precip1_depth: missing_value=None, _FillValue=-1e+30
precip2_depth: missing_value=None, _FillValue=-1e+30
precip3_depth: missing_value=None, _FillValue=-1e+30
precip6_depth: missing_value=None, _FillValue=-1e+30
precip9_depth: missing_value=None, _FillValue=-1e+30
precip12_depth: missing_value=None, _FillValue=-1e+30
precip15_depth: missing_value=None, _FillValue=-1e+30
precip18_depth: missing_value=None, _FillValue=-1e+30
precip24_depth: missing_value=None, _FillValue=-1e+30
cloud_base: missing

In [34]:
ds1

<xarray.Dataset> Size: 4GB
Dimensions:                (station: 10, time: 473329, test: 71, flagged: 19,
                            reporting_v: 19, reporting_t: 1116, reporting_2: 2,
                            coordinate_length: 1)
Coordinates:
  * time                   (time) datetime64[ns] 4MB 1970-01-01 ... 2023-12-31
    longitude              (station, coordinate_length) float64 80B dask.array<chunksize=(1, 1), meta=np.ndarray>
    latitude               (station, coordinate_length) float64 80B dask.array<chunksize=(1, 1), meta=np.ndarray>
    elevation              (station, coordinate_length) float64 80B dask.array<chunksize=(1, 1), meta=np.ndarray>
  * station                (station) <U12 480B '010280-99999' ... '010550-99999'
Dimensions without coordinates: test, flagged, reporting_v, reporting_t,
                                reporting_2, coordinate_length
Data variables: (12/26)
    station_id             (station) |S12 120B dask.array<chunksize=(1,), meta=np.ndarray>
    temperatures           (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    dewpoints              (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    slp                    (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    stnlp                  (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    windspeeds             (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    ...                     ...
    cloud_base             (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    wind_gust              (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    past_sigwx1            (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    quality_control_flags  (station, time, test) float64 3GB dask.array<chunksize=(1, 473329, 71), meta=np.ndarray>
    flagged_obs            (station, time, flagged) float64 719MB dask.array<chunksize=(1, 473329, 19), meta=np.ndarray>
    reporting_stats        (station, reporting_v, reporting_t, reporting_2) float64 3MB dask.array<chunksize=(1, 19, 1116, 2), meta=np.ndarray>
Attributes: (12/39)
    title:                       HadISD
    institution:                 Met Office Hadley Centre, Exeter, UK
    source:                      HadISD data product
    references:                  Dunn, 2019, Met Office Hadley Centre Technic...
    creator_name:                Robert Dunn
    creator_url:                 www.metoffice.gov.uk
    ...                          ...
    station_information:         Where station is a composite the station id ...
    Conventions:                 CF-1.6
    Metadata_Conventions:        Unidata Dataset Discovery v1.0, CF Discrete ...
    featureType:                 timeSeries
    processing_date:             08-Jan-2024
    history:                     Created by mk_netcdf_files.py \nDuplicate Mo...

In [35]:
for var in ds1.data_vars:
    mv = ds1[var].attrs.get("missing_value")
    fv = ds1[var].encoding.get("_FillValue")
    if mv is not None or fv is not None:
        print(f"{var}: missing_value={mv}, _FillValue={fv}")

temperatures: missing_value=None, _FillValue=-1e+30
dewpoints: missing_value=None, _FillValue=-1e+30
slp: missing_value=None, _FillValue=-1e+30
stnlp: missing_value=None, _FillValue=-1e+30
windspeeds: missing_value=None, _FillValue=-1e+30
winddirs: missing_value=None, _FillValue=-999
total_cloud_cover: missing_value=None, _FillValue=-999
low_cloud_cover: missing_value=None, _FillValue=-999
mid_cloud_cover: missing_value=None, _FillValue=-999
high_cloud_cover: missing_value=None, _FillValue=-999
precip1_depth: missing_value=None, _FillValue=-1e+30
precip2_depth: missing_value=None, _FillValue=-1e+30
precip3_depth: missing_value=None, _FillValue=-1e+30
precip6_depth: missing_value=None, _FillValue=-1e+30
precip9_depth: missing_value=None, _FillValue=-1e+30
precip12_depth: missing_value=None, _FillValue=-1e+30
precip15_depth: missing_value=None, _FillValue=-1e+30
precip18_depth: missing_value=None, _FillValue=-1e+30
precip24_depth: missing_value=None, _FillValue=-1e+30
cloud_base: missing

In [25]:
def clean_fillvalue_attrs(ds):
    for var in ds.data_vars:
        enc = ds[var].encoding
        # Remove conflicting attributes
        if '_FillValue' in enc and 'missing_value' in enc:
            if enc['_FillValue'] != enc['missing_value']:
                # Prefer np.nan as _FillValue, remove missing_value
                ds[var].encoding['_FillValue'] = np.nan
                del ds[var].encoding['missing_value']
        # If only missing_value exists, convert to _FillValue
        elif 'missing_value' in enc:
            ds[var].encoding['_FillValue'] = enc['missing_value']
            del ds[var].encoding['missing_value']
    return ds

In [ ]:
ds_cleaned = clean_fillvalue_attrs(ds)
ds

In [23]:
for var in batch_ds.data_vars:
    print(f"{var}: {batch_ds[var].chunks}")

station_id: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1),)
temperatures: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
dewpoints: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
slp: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
stnlp: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
windspeeds: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
winddirs: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
total_cloud_cover: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
low_cloud_cover: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
mid_cloud_cover: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
high_cloud_cover: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
precip1_depth: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
precip2_depth: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
precip3_depth: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
precip6_depth: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
precip9_depth: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
precip12_depth: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (473329,))
precip15_depth: ((1, 1, 1, 1,

In [ ]:
    # Write to Zarr
with ProgressBar():
    if i == 0:
        try:
            batch_ds.to_zarr(zarr_output_dir, mode="w")
            print("Initial Zarr store created.")
        except Exception as e:
            print(f"Error creating Zarr store: {e}")
    else:
        try:
            batch_ds.to_zarr(zarr_output_dir, mode="a", append_dim="station")
        except Exception as e:
            print(f"Error appending to Zarr: {e}")

# Free memory
del batch_ds, batch_datasets

print("Zarr store written successfully!")


In [52]:
# Open zarr store using xarray
ds_combined = xr.open_zarr(zarr_output_dir)
print(ds_combined)

<xarray.Dataset> Size: 8GB
Dimensions:                (station: 20, time: 473329, coordinate_length: 1,
                            flagged: 19, test: 71, reporting_v: 19,
                            reporting_t: 1116, reporting_2: 2)
Coordinates:
    elevation              (station, coordinate_length) float64 160B dask.array<chunksize=(1, 1), meta=np.ndarray>
    latitude               (station, coordinate_length) float64 160B dask.array<chunksize=(1, 1), meta=np.ndarray>
    longitude              (station, coordinate_length) float64 160B dask.array<chunksize=(1, 1), meta=np.ndarray>
  * station                (station) <U12 960B '010010-99999' ... '010550-99999'
  * time                   (time) datetime64[ns] 4MB 1970-01-01 ... 2023-12-31
Dimensions without coordinates: coordinate_length, flagged, test, reporting_v,
                                reporting_t, reporting_2
Data variables: (12/26)
    cloud_base             (station, time) float64 76MB dask.array<chunksize=(1, 473329

In [47]:
def drop_fillvalue_attrs(ds):
    for var in ds.data_vars:
        ds[var].encoding.pop('missing_value', None)
        ds[var].encoding.pop('_FillValue', None)
    return ds

ds_combined = drop_fillvalue_attrs(ds_combined)
ds_combined

<xarray.Dataset> Size: 4GB
Dimensions:                (station: 10, time: 473329, coordinate_length: 1,
                            flagged: 19, test: 71, reporting_v: 19,
                            reporting_t: 1116, reporting_2: 2)
Coordinates:
    elevation              (station, coordinate_length) float64 80B dask.array<chunksize=(1, 1), meta=np.ndarray>
    latitude               (station, coordinate_length) float64 80B dask.array<chunksize=(1, 1), meta=np.ndarray>
    longitude              (station, coordinate_length) float64 80B dask.array<chunksize=(1, 1), meta=np.ndarray>
  * station                (station) <U12 480B '010010-99999' ... '010260-99999'
  * time                   (time) datetime64[ns] 4MB 1970-01-01 ... 2023-12-31
Dimensions without coordinates: coordinate_length, flagged, test, reporting_v,
                                reporting_t, reporting_2
Data variables: (12/26)
    cloud_base             (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    dewpoints              (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    flagged_obs            (station, time, flagged) float64 719MB dask.array<chunksize=(1, 473329, 19), meta=np.ndarray>
    high_cloud_cover       (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    low_cloud_cover        (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    mid_cloud_cover        (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    ...                     ...
    stnlp                  (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    temperatures           (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    total_cloud_cover      (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    wind_gust              (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    winddirs               (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
    windspeeds             (station, time) float64 38MB dask.array<chunksize=(1, 473329), meta=np.ndarray>
Attributes: (12/39)
    Conventions:                 CF-1.6
    Metadata_Conventions:        Unidata Dataset Discovery v1.0, CF Discrete ...
    acknowledgement:             RJHD was supported by the Joint BEIS/Defra M...
    cdm_data_type:               station
    creator_email:               robert.dunn@metoffice.gov.uk
    creator_name:                Robert Dunn
    ...                          ...
    station_id:                  010010-99999
    station_information:         Where station is a composite the station id ...
    summary:                     Quality-controlled, sub-daily, station datas...
    time_coverage_end:           2023-12-31T23:00Z
    time_coverage_start:         1931-01-01T06:00Z
    title:                       HadISD

In [48]:
for var in ds_combined.data_vars:
    mv = ds_combined[var].attrs.get("missing_value")
    fv = ds_combined[var].encoding.get("_FillValue")
    if mv is not None or fv is not None:
        print(f"{var}: missing_value={mv}, _FillValue={fv}")

In [ ]:
print(ds_combined)

In [ ]:
from pyearthtools.tutorial.HadisdDataClass import HadISDIndex

In [ ]:
hadisd = HadISDIndex()
all_stations = hadisd.get_all_station_ids()
all_stations_ordered = sorted(all_stations)
print(f"Total number of stations: {len(all_stations_ordered)}")

hadisd.station = ["010010-99999"]
hadisd.station = all_stations_ordered[:10]

In [ ]:
paths = hadisd.filesystem()
paths